In [ ]:
# pcsk_2NmpqW_DCpwKzkz4X7SgURu2fXN1DuDJYehiDm6kUC8CJpYpgP51VwhXUps1TUCrFygaqG

In [5]:
from pinecone import Pinecone,ServerlessSpec

In [2]:
pc = Pinecone(api_key = 'pcsk_2NmpqW_DCpwKzkz4X7SgURu2fXN1DuDJYehiDm6kUC8CJpYpgP51VwhXUps1TUCrFygaqG')

In [ ]:
# 准备元数据
items = [
    {"type": "phone", "id": "用户A", "number": 13800001234},
    {"type": "phone", "id": "用户B", "number": 13800005678},
    {"type": "order", "id": "订单1001", "number": 203011010001},
    {"type": "order", "id": "订单1002", "number": 203011010123},
    {"type": "order", "id": "订单2001", "number": 203012150045},
    {"type": "phone", "id": "用户C", "number": 13912345678},
    {"type": "phone", "id": "用户D", "number": 13798765432},
    {"type": "order", "id": "订单3001", "number": 205001020333},
    {"type": "order", "id": "订单3002", "number": 205001020777},
    {"type": "phone", "id": "用户E", "number": 13622223333},
]

In [ ]:
# 创建索引
index_name = 'number-vectors'
dimension = 1

# 获取当前索引列表名称
existing_indexs = [idx.name for idx in pc.list_indexes()]
# 判断是否存在
if index_name  in existing_indexs:
    # 删除索引
    pc.delete_index(index_name)

# 创建索引
pc.create_index(
    name=index_name,    # 索引名称
    dimension=dimension, #向量维度
    metric='euclidean',  # 向量距离度量方法
    spec = ServerlessSpec(cloud='aws',region='us-east-1')
)
index = pc.Index(index_name)

In [ ]:
vectors = []
for i, item in enumerate(items, start=1):
    number_vector = [float(item['number'])]  # 注意：这是 1 维向量
    vectors.append({
        "id": str(i),                         # ← 主键（唯一标识）
        "values": number_vector,              # ← 向量数据（自动建索引）
        "metadata": {                         # ← 标量数据（不建索引，仅存储）
            "type": item['type'],
            "biz_id": item['id'],
            "number": item['number']
        }
    })

index.upsert(vectors)  # 插入的是「向量 + 元数据」

已向 Pinecone 索引写入 10 条 1 维向量。


In [10]:
# 查询数据
query_number = 205001020500
query_vector = [float(query_number)]

k = 5

results = index.query(
    vector=query_vector,      # 要查询的数据
    top_k= k,                 #  获取前几条数据
    include_metadata=True,    # 是否获取元数据
    include_values=False      # 是否返回实际的向量值
)
results

QueryResponse(matches=[{'id': '9',
 'metadata': {'biz_id': '订单3002', 'number': 205001020777, 'type': 'order'},
 'score': 0.0,
 'values': []}, {'id': '8',
 'metadata': {'biz_id': '订单3001', 'number': 205001020333, 'type': 'order'},
 'score': 0.0,
 'values': []}, {'id': '5',
 'metadata': {'biz_id': '订单2001', 'number': 203012150045, 'type': 'order'},
 'score': 3.95416047e+18,
 'values': []}, {'id': '4',
 'metadata': {'biz_id': '订单1002', 'number': 203011010123, 'type': 'order'},
 'score': 3.96316767e+18,
 'values': []}, {'id': '3',
 'metadata': {'biz_id': '订单1001', 'number': 203011010001, 'type': 'order'},
 'score': 3.96316767e+18,
 'values': []}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Mon, 01 Dec 2025 10:18:40 GMT', 'content-type': 'application/json', 'content-length': '629', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '50', 'x-pinecone-request-id': '8647824077755820001', 'x-envoy-upstream-service

In [11]:
print("\n查询数字：", query_number)
print(f"Top-{k} 最相近数字（按欧式距离，越小越相近）：")

for rank , match in enumerate(results.matches,start =1):
    meta = match.metadata or {}
    print(
        f"{rank}. 距离={match.score:.4f} | 类型={meta.get('type')} | 标识={meta.get('biz_id')} | 数字={meta.get('number')}"
    )


查询数字： 205001020500
Top-5 最相近数字（按欧式距离，越小越相近）：
1. 距离=0.0000 | 类型=order | 标识=订单3002 | 数字=205001020777
2. 距离=0.0000 | 类型=order | 标识=订单3001 | 数字=205001020333
3. 距离=3954160470000000000.0000 | 类型=order | 标识=订单2001 | 数字=203012150045
4. 距离=3963167670000000000.0000 | 类型=order | 标识=订单1002 | 数字=203011010123
5. 距离=3963167670000000000.0000 | 类型=order | 标识=订单1001 | 数字=203011010001
